In [1]:
# %pip install -r ../requirements.txt
# # !! be aware !!
# # FastPFOR compression library python binding
# # only compiles with gcc -> on windows, requires mingw compiler
# #   as per detail:
# #   https://github.com/fast-pack/FastPFOR?tab=readme-ov-file#software-requirements
# # testing so far is only done on linux
# # if there are issues on windows...
# #   it is planned to provide a
# #   precompiled dll interface to FastPFOR

In [2]:
from io import BytesIO
import sys
sys.path.append('../')
from mdfc import (
    MDFCompressor, MDFDecompressor
)

import asammdf, numpy as np


In [3]:
# parameters to generate sample MF4 file
example_mdf_params = dict(
    # random int/float data
    include_random_ints=False,
    include_random_floats=False,
    # non-random int/float data
    include_sine_waves=True,  # False,  # True,
    include_keepalives=True,  # True
    # extent & grouping parameters
    time_s=3600*4,
    sample_intervals_ms=(500,100, 50, 10),
    channels_per_interval=1,
    channels_per_group=1,

    # adding some jitter to each time
    jitter_time=0.05
)

In [4]:
# uncompressed MDF file
from sample_data.generate_sample_data import generate_sample_file
MDF_FIL = BytesIO()
generate_sample_file(
    MDF_FIL, 
    compression=False,
    **example_mdf_params
)
MDF_FIL.seek(0); pass

8 total groups are generated


In [5]:
# size of uncompressed MDF file in MB
uncompressed_mdf_total_size = MDF_FIL.__sizeof__()
print(
    f'{uncompressed_mdf_total_size/1000/1000:.2f} '
    'MB Uncompressed MDF File'
)

59.31 MB Uncompressed MDF File


In [6]:
# compare against deflate-2
DEFLATE_MDF_FIL = BytesIO()
with (
    asammdf.MDF(MDF_FIL) as mdf_uc,
    asammdf.MDF() as mdf_c
):
    for gg in mdf_uc.iter_groups(
        time_from_zero=False,
        raw=True,
        ignore_value2text_conversions=True,
    ):
        mdf_c.append(gg)
    mdf_c.save(DEFLATE_MDF_FIL, compression=2)


In [7]:
# size of deflated MDF file in MB
deflate_mdf_total_size = DEFLATE_MDF_FIL.__sizeof__()
print(
    f'{deflate_mdf_total_size/1000/1000:.2f} '
    'MB Deflate2 MDF File'
)

28.35 MB Deflate2 MDF File


In [8]:
# ratio of deflate vs uncompressed
print(
    f'{uncompressed_mdf_total_size/deflate_mdf_total_size:.3f} '
    'CR using Deflate (MDF Standard)'
)

2.092 CR using Deflate (MDF Standard)


In [9]:
# configurable parameters for mdfc compression,
# which is just for lossy float compression
# for lossless fp compression, set:
#   tolerance, significands, minimum_tolerance
#   = -1  (the default values)
#   TODO allow some false-y value also, or None
#        but presently, it would raise value error :(
# in this example we can use these lossy params:
lossy_fp_params = dict(
    # significands = 20,  # big number just falls back to min tolerance
    # ^ meaning: 
    #   n additional digits
    #   after the significance
    #   of the smallest value
    #       uniquely for each channel
    #   eg: 
    #       if significands = 3,
    #       if channel A min_value == 1e-5,
    #       then channel A tolerance =  1e-8
    # minimum_tolerance = 1e-4,
    # ^ meaning:
    #   minimum tolerance for all channels
    
)
lossy_time_params = dict(
    # highest precision -> 1 ns
    # real-world data might be OK at 10us
    time_resolution = '1ns'
)

In [10]:
# test params
DO_TEST_COMPRESSION   = True
DO_TEST_DECOMPRESSION = True

In [11]:
# %%timeit
# test compression
if DO_TEST_COMPRESSION:
    MDFC_FIL = BytesIO()
    MDFC_FIL.seek(0); MDF_FIL.seek(0); pass
    with (
        asammdf.MDF(MDF_FIL) as mdf_fil,
        MDFCompressor(MDFC_FIL, close_file_on_exit=False) as mdfc_fil
    ):
        mdfc_fil.compress_all_groups(
            mdf_fil,
            on_error='warn',
            **lossy_fp_params,
            **lossy_time_params,
        )
        # presently, must call finish function,
        #   TODO it should be done on a (successful?) __exit__
        mdfc_fil.finish()
    MDFC_FIL.seek(0); MDF_FIL.seek(0); pass

using lossless compress_f, be aware proper decompression is not implemented yet!
using lossless compress_f, be aware proper decompression is not implemented yet!
using lossless compress_f, be aware proper decompression is not implemented yet!
using lossless compress_f, be aware proper decompression is not implemented yet!
1878 bytes of metadata


In [12]:
mdfc_total_size = MDFC_FIL.__sizeof__()
print(
    f'{mdfc_total_size/1000/1000:.2f} '
    'MB MDFC File'
)

23.23 MB MDFC File


In [13]:
# ratio of mdfc vs uncompressed
cr_vs_uncomp = (MDFC_FIL.__sizeof__() / MDF_FIL.__sizeof__())
print(
     'Overall compression ratio vs uncompressed is '
    f'{cr_vs_uncomp:.3f}, or {1/cr_vs_uncomp:.2f}x'
)

Overall compression ratio vs uncompressed is 0.392, or 2.55x


In [14]:
# ratio of mdfc vs deflate
cr_vs_deflate = (MDFC_FIL.__sizeof__() / DEFLATE_MDF_FIL.__sizeof__())
print(
     'Overall compression ratio vs Deflate-2 is '
    f'{cr_vs_deflate:.3f}, or {1/cr_vs_deflate:.2f}x'
)

Overall compression ratio vs Deflate-2 is 0.820, or 1.22x


In [15]:
# does it help vs deflate-2? :) ... or :(

In [16]:
# %%timeit
# execute decompression & compare against original
err_sig = None
err_sig_orig = None
def decompress_and_compare(sn, mdfc_fil, mdf_fil):
    global err_sig
    global err_sig_orig
    # decompress the signal from mdfc
    # and compare it against the signal in mdf
    original_sig = mdf_fil.select([sn], raw=True)[0]
    original_timestamps = original_sig.timestamps
    original_samples = original_sig.samples
    
    # decompress mdfc signal
    res = mdfc_fil.decompress_signal(sn)

    # assert all close timestamps and values
    # timestamps... may have some minor losses
    #   due to float->scaleup->int on compression
    #   i think it should be understood that the retention
    #   should be based on the scale
    try:
        assert np.allclose(
            original_timestamps,
            res.timestamps,
            # the "scaleup" applied on compression
            #   fp inaccuracies may be expected
            #   past this precision
            atol=(1/mdfc_fil.time_metadata[-1][0][1])
        ), f"{sn} timestamps not allclose!? :("
        assert np.allclose(
            original_samples,
            res.samples,
            # tolerance specification for float case
            # TODO perhaps this should be derived
            #   from compression metadata,
            #   ie the tolerance value used
            # occasionally this is not right
            atol=lossy_fp_params.get('minimum_tolerance', 1e-20)
        ), f"{sn} samples not allclose!? :("
    except:
        err_sig = res
        err_sig_orig = original_sig
        raise


if DO_TEST_DECOMPRESSION:
    MDFC_FIL.seek(0); MDF_FIL.seek(0); pass
    with (
        asammdf.MDF(MDF_FIL) as mdf_fil,
        MDFDecompressor(MDFC_FIL, 
                        close_file_on_exit=False,
                        load_time_axis_on_enter=False) as mdfc_fil
    ):
        mdfc_fil.decompress_time()
        try:
            # test signal decompression
            for sn in mdf_fil.channels_db.keys():
                if sn == 'time': continue  #
                print(f'on {sn}')
                decompress_and_compare(sn, mdfc_fil, mdf_fil)
                print(f'pass {sn}')
        except KeyError:
            print(f'{sn} found in MDF but not in compressed file...')
            raise  # ?
        else:
            print("All signals have passed decompression check :)")
    MDFC_FIL.seek(0); MDF_FIL.seek(0); pass

on sine_500ms_1
attempt decompress lossless, if you are reading this message, please enhance the decompression implementation for fp :)
pass sine_500ms_1
on sine_100ms_2
attempt decompress lossless, if you are reading this message, please enhance the decompression implementation for fp :)
pass sine_100ms_2
on sine_50ms_3
attempt decompress lossless, if you are reading this message, please enhance the decompression implementation for fp :)
pass sine_50ms_3
on sine_10ms_4
attempt decompress lossless, if you are reading this message, please enhance the decompression implementation for fp :)
pass sine_10ms_4
on counter_500ms_1
pass counter_500ms_1
on counter_100ms_2
pass counter_100ms_2
on counter_50ms_3
pass counter_50ms_3
on counter_10ms_4
pass counter_10ms_4
All signals have passed decompression check :)


In [17]:
# lets do some time checks...
test_names = [
    # ... specific signal names...
    # 'acceleration_id'
    # 'x_value'
]
test_names = None  # all signals

In [18]:
%%timeit
# testing the speed of reading MDF (without compression)
MDF_FIL.seek(0)
with (
    asammdf.MDF(MDF_FIL) as mfil,
):
    if test_names is None:
        sigs = [
            sig_name # (sig_name, *chan_info) 
            for sig_name, chan_info in mfil.channels_db.items()
            if sig_name != 'time'
        ]
    else:
        sigs = test_names
    for sig_sel in sigs:
        sig = mfil.select([sig_sel], raw=True)[0]

64.8 ms ± 2.02 ms per loop (mean ± std. dev. of 7 runs, 10 loops each)


In [19]:
# %%timeit
# # testing the speed of reading MDF (with deflate compression)
# DEFLATE_MDF_FIL.seek(0)
# with (
#     asammdf.MDF(DEFLATE_MDF_FIL) as mfil,
# ):
#     if test_names is None:
#         sigs = [
#             sig_name # (sig_name, *chan_info) 
#             for sig_name, chan_info in mfil.channels_db.items()
#             if sig_name != 'time'
#         ]
#     else:
#         sigs = test_names
#     for sig_sel in sigs:
#         sig = mfil.select([sig_sel], raw=True)[0]

In [20]:
%%timeit
# testing the speed of reading the MDFC compressed file
MDFC_FIL.seek(0)
with MDFDecompressor(MDFC_FIL, close_file_on_exit=False) as dfil:
    if test_names is None:
        sigs = dfil.channame_to_group.keys()
    else:
        sigs = test_names
    for sn in sigs:
        res = dfil.decompress_signal(sn)

attempt decompress lossless, if you are reading this message, please enhance the decompression implementation for fp :)
attempt decompress lossless, if you are reading this message, please enhance the decompression implementation for fp :)
attempt decompress lossless, if you are reading this message, please enhance the decompression implementation for fp :)
attempt decompress lossless, if you are reading this message, please enhance the decompression implementation for fp :)
attempt decompress lossless, if you are reading this message, please enhance the decompression implementation for fp :)
attempt decompress lossless, if you are reading this message, please enhance the decompression implementation for fp :)
attempt decompress lossless, if you are reading this message, please enhance the decompression implementation for fp :)
attempt decompress lossless, if you are reading this message, please enhance the decompression implementation for fp :)
attempt decompress lossless, if you are 

In [21]:
# end time checks :) ... or :(

In [ ]:
# thanks for playing!